# Kombinierte Labels: Standard Normal Variate

**Hypothese:** Die Kurvenform der Reflektanzspektren trennt Crop/Stage robuster als ihr absolutes Helligkeitsniveau. SNV normalisiert deshalb jedes interpolierte Spektrum auf Mittelwert null und Standardabweichung eins; AEZ und Month umgehen diese Transformation.

**Entscheidung:** Die RBF-SVM der Rohspektren-Referenz bleibt mit `C=100.0` und `gamma=0.003` fest. Nur wenn SNV auf den fünf Trainingsfolds die kombinierte CV-BAcc der Referenz (`0.8473`) übertrifft, wird diese Variante einmal auf dem gemeinsamen Validierungssplit bewertet.

In [ ]:
%load_ext autoreload
%autoreload 2

from awp2.experiment import TuneConfig, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

REFERENCE_CV_BACC_COMBINED = 0.8473
REFERENCE_SVM_PARAMS = {
    "estimator__C": 100.0,
    "estimator__gamma": 0.003,
}
snv_preprocessing = PreprocessingConfig(
    scale=True,
    use_meta=True,
    use_spectral_snv=True,
)

## 1. SNV ausschließlich auf den Trainingsfolds prüfen

Interpolation, SNV und Skalierung liegen gemeinsam in der Pipeline. Damit wird keine Eigenschaft des Validierungssplits zur Vorverarbeitung oder zur Modellwahl verwendet.

In [ ]:
tuned = tune(
    make_combined_svm(),
    TuneConfig(
        name="combined_svm_snv_search",
        approach="combined",
        description=(
            "RBF-SVM mit Standard Normal Variate nur auf interpolierten Spektralbändern; "
            "AEZ und Month bleiben unverändert im Metadatenzweig."
        ),
        preprocessing=snv_preprocessing,
        param_grid={key: [value] for key, value in REFERENCE_SVM_PARAMS.items()},
    ),
)

{
    "CV BAcc kombiniert (SNV)": tuned.best_score,
    "CV BAcc kombiniert (Referenz)": REFERENCE_CV_BACC_COMBINED,
    "Parameter": tuned.best_params,
}

## 2. Entscheidung nach der Cross-Validation

SNV wird nur bei einem CV-Gewinn auf dem Holdout bewertet. Ein niedrigerer CV-Wert beendet den Versuch ohne Validierung, damit der gemeinsame Holdout für eine spätere, neue Hypothese unangetastet bleibt.

In [ ]:
from IPython.display import display

from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import plot_tuning_candidates, save_doc_figure

cv_difference = tuned.best_score - REFERENCE_CV_BACC_COMBINED
assert cv_difference < 0

decision = {
    "CV-Differenz zu Rohspektren": cv_difference,
    "Holdout bewertet": False,
    "Tuning-Run": tuned.run_id,
}
display(decision)

tuning_ax = plot_tuning_candidates(tuned.candidates, color_by="estimator__C")
tuning_ax.set_title("Kombinierte Labels: RBF-SVM mit SNV")
display(tuning_ax.figure)
save_doc_figure(tuning_ax.figure, "combined_svm_snv_tuning", MODEL_DOCS_FIGURES_DIR)

## Fazit

Die kombinierte RBF-SVM mit SNV erreicht in der Trainings-CV `0.8233` gegenüber `0.8473` für die Rohspektren-Referenz. Die Hypothese wird nicht gestützt; der gemeinsame Validierungssplit bleibt für eine spätere, unabhängige Hypothese unberührt.